# Fig. 3: Seeded cumulant expansion in the $(N_c, N_g)$ plane

Same model and grid as `2_cumulants_Nc_Ng.ipynb`, with a coherent seed $\Lambda$ driving the squeezed mode. At each point the seed amplitude is increased until a coherent (lasing) solution is found, and its quadrature fluctuations $\Delta\hat X^2$ are saved (**panel e**).

**Inputs** (`../data/Nc_Ng/`, from notebook 2): `n.npy`, `iterations.npy`.
**Outputs** (`../data/Nc_Ng/`): `dX2_seed.npy`.

In [ ]:
using QuantumCumulants
using ModelingToolkit
using OrdinaryDiffEq
using WignerSymbols
using LinearAlgebra
using ProgressMeter
using NPZ

In [ ]:
Threads.nthreads()

In [ ]:
function dipole_me(F,Fp,m,p) # Dipole matrix element of the D1 line between |F,m> and |F',m+p> (Rb87, I = 3/2)
    
    J_g = 1/2
    J_e = 1/2
    I = 3/2

    sixj = Float64.(wigner6j(J_g, J_e, 1, Fp, F, I))
    cg = Float64.(clebschgordan(Fp, m+p, 1, -p, F, m))

    return (-1)^(Fp+J_g+1+I)*sqrt((2*Fp+1)*(2*J_g+1))*cg*sixj
end

## Parameters

In [ ]:
# Grid
num_Nc = 150
num_Nl = 150

# Rb87 level structure (B in G; angular frequencies, with the frequency in GHz)
B = 2.7
gamma = 2*pi*5.75*1e-3 # D1 linewidth
gamma_D2 = 2*pi*6.06*1e-3 # D2 linewidth
w_t = 2*pi*377107.463380
w_t_D2 = 2*pi*384230.48446
w_e = 2*pi*(6.834-3*0.7*1e-3*B)
w_s = w_t+2*pi*0.30544+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*2*0.23*1e-3*B
w_r = w_t-2*pi*0.50906+2*pi*4.271677-2*pi*0.7*1e-3*B+2*pi*0.23*1e-3*B
w_u = w_t_D2-2*pi*0.22985+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*0.93*1e-3*B

# Number of cooling (N_atom_I = N_c) and gain (N_atom_II = N_g) atoms
N_atom_I = logrange(1e2,2e4,num_Nc)
N_atom_II = logrange(10,2e4,num_Nl)
N_atom = N_atom_I .+ N_atom_II'

# Auxiliary cavity mode b
gb = 3*2*pi*1e-3
omega = 40*gamma_D2
w_b = w_u + 2*2pi
kappa_b = 20*2*pi*1e-3

# Main cavity mode a
kappa = 0.07*2*pi*1e-3
gr = 0.78*2*pi*1e-3
gs = gr*abs(dipole_me(2,2,-2,0)/dipole_me(1,1,-1,0))
r = 0.5 # Squeezing parameter
eps_cav = 1*1e-3*2*pi
w_cav = (1/3)*(w_e+4*w_r-w_s)+eps_cav

In [ ]:
function P_select(P_pk)

    P_cap = 200*1e-6*2pi

    if P_pk<P_cap
        return P_pk
    else
        return P_cap
    end
end

## Adiabatic elimination

Laser frequencies, rotating-frame detunings and effective couplings obtained after eliminating the excited levels $u$, $r$, $s$ and then the modes $b$ and $s$ (steps 1–7), and the effective pumping, decay and dephasing rates of the atoms (step 8).

In [ ]:
# Step 1: find convenient frequencies for the lasers

w_lr = (-w_cav+3*w_r-w_s).-sqrt.(4*gr^2*N_atom.+(w_cav-w_r)^2)
w_ls = -2*w_lr.+(4*w_r-w_s)
w_l = 0.5*(w_b.+w_lr.-w_ls.+w_u.+sqrt.(4*gb^2*N_atom_I.+(w_b-w_u)^2 .-omega^2))

# Step 2: move to rotating frame

delta_b = w_b .- w_l .- (w_ls.-w_lr)./2
delta_u = w_u .- w_l .- (w_ls.-w_lr)./2
delta_1 = w_e.-(w_ls.-w_lr)./2
delta_s = w_s.-w_ls
delta_r = w_r.-(w_ls.+w_lr)./2
delta_cav = w_cav.-(w_ls.+w_lr)./2

# Step 3: eliminate level u

w_0_b = -0.25.*(omega.^2)./delta_u
w_c_b = delta_b.-(N_atom_I*gb^2)./delta_u
chi_b = (gb^2)./delta_u
g = -omega.*gb./(2*delta_u)

# Step 4: find Rabi frequencies

omega_r = sqrt.((w_ls.-2*w_e.-w_lr)./(1 ./(w_lr.+w_ls.-2*w_r).-2*tanh.(r).^2 ./(w_ls.-w_s)))
omega_s = (gr/gs)*omega_r.*tanh.(r).*(delta_s./delta_r)

# Step 5: eliminate levels r and s

w_0 = delta_1.-0.25.*((omega_r.^2)./delta_r.-(omega_s.^2)./delta_s)
w_c = delta_cav.-(N_atom*gr^2)./delta_r
chi = (gr^2)./delta_r.-(gs^2)./delta_s
lambda_r = -omega_r.*gr./(2*delta_r)
lambda_s = -omega_s.*gs./(2*delta_s)
g_02 = (lambda_r.^2).-lambda_s.^2

alfa = omega_r./(2*delta_r)
beta = gr./delta_r
delta = omega_s./(2*delta_s)
xi = gs./delta_s
alfa_b = omega./(2*delta_u)
beta_b = gb./delta_u

dec_eff = gamma*dipole_me(1,1,-1,0)^2*alfa.^2
P_eff = gamma*dipole_me(2,2,-2,0)^2*delta.^2
dep_eff = (gamma*dipole_me(1,2,-1,-1)^2*delta.^2 .+ gamma*dipole_me(2,1,-2,+1)^2*alfa.^2)/4
dec_eff_b = gamma_D2*(5/12)*alfa_b.^2
dep_eff_b = gamma_D2*(1/10)*alfa_b.^2/4

# Step 6: Eliminate mode b

dec = (g.^2*kappa_b)./((w_c_b.-w_0_b).^2 .+ kappa_b^2/4)
delta_lamb = w_0_b .+ w_0 .+ N_atom_I.*(g.^2).*(w_c_b.-w_0_b)./((kappa_b^2/4).+(w_c_b.-w_0_b).^2) # coefficient of s's term

# Step 7: Eliminate mode s

dec_f = dec.*g_02./(dec.^2/4 .+ (delta_lamb./N_atom_I).^2)
delta_lamb_f = (g_02.*delta_lamb.*N_atom_I)./((delta_lamb.^2).+(N_atom_I.*dec/2).^2)

# Step 8: include pumping and dephasing

dec_T = dec_eff
dep_T = dep_eff

Γ_c = (4*g_02./(kappa.+dec_f)) # Purcell decay of qubits
P = P_select.((N_atom_II'.*Γ_c)/2 .-dec_T.-2*dep_T)

P_T = P.+P_eff

C = (2*N_atom_II'.*g_02)./(((P_T.+dec_T)/2 .+ 2*dep_T).*(dec_f.+kappa))./(1 .+(4*delta_lamb_f.^2)./((kappa.+dec_f).+(P_T.+dec_T.+4*dep_T)).^2)
n_q = (P_T.-dec_T)./(2*(kappa.+dec_f)).*N_atom_II'
control = P_T./(kappa.+dec_f).-((P_T.+dec_T)./(2*(kappa.+dec_f))).*(1 .+ 1 ./ C)

## Cumulant expansion

Squeezed cavity mode $a_s$ (with $a = a_s\cosh r - a_s^\dagger\sinh r$), auxiliary mode $b$, $N_c$ cooling atoms and $N_g$ gain atoms, after the adiabatic elimination of the excited levels. Second-order cumulant expansion with QuantumCumulants.jl. The coherent seed enters as $\Lambda\,(i a_s - i a_s^\dagger)$.

In [ ]:
@rnumbers Δ_cav_b Δ_0_b g_b G χ_b κ κ_b R Nc Nl pc γ_ϕc γc pl γ_ϕl γl Λ # Parameters

h_cav = FockSpace(:cavity) # Hilbert space of the cavity
h_atom_C = NLevelSpace(:atom_c,(:g,:e)) # Hilbert space of the atom
h_atom_L = NLevelSpace(:atom_l,(:g,:e)) # Hilbert space of the atom
h_cav_b = FockSpace(:cavity_b)
h = h_cav⊗h_atom_C⊗h_cav_b⊗h_atom_L # Complete Hilbert space

σc(x,y,k) = IndexedOperator(Transition(h, :σc, x, y, 2), k) # Operators
σl(x,y,k) = IndexedOperator(Transition(h, :σl, x, y, 4), k) # Operators
as = Destroy(h,:as,1)
a = as*cosh(R)-as'*sinh(R)
b = Destroy(h,:b,3)

ic = Index(h, :ic, Nc, h_atom_C)
jc = Index(h, :jc, Nc, h_atom_C)
il = Index(h, :il, Nl, h_atom_L)
jl = Index(h, :jl, Nl, h_atom_L)

Hc = Δ_cav_b*b'*b + ∑(Δ_0_b*σc(:e,:g,ic)*σc(:g,:e,ic) + G*(as*σc(:e,:g,ic)+as'*σc(:g,:e,ic)) + g_b*(b*σc(:e,:g,ic)+b'*σc(:g,:e,ic)) + χ_b*σc(:e,:g,ic)*σc(:g,:e,ic)*b'*b,ic) # Hamiltonian
Hl = ∑(G*(as*σl(:e,:g,il)+as'*σl(:g,:e,il)),il)

H = Hc + Hl + Λ*(im*as-im*as')

D = [a;b;σc(:e,:g,ic);σc(:e,:e,ic)-σc(:g,:g,ic);σc(:g,:e,ic);σl(:e,:g,il);σl(:e,:e,il)-σl(:g,:g,il);σl(:g,:e,il)]
rates = [κ; κ_b; pc; γ_ϕc; γc; pl; γ_ϕl; γl]

In [ ]:
ops = [as] # Operators whose equations I want to derive
eqs = meanfield(ops,H,D;rates=rates,order=2) # Derive set of equations and factorize correlators for closing the set of equations
me_comp = complete!(eqs) # Automatically complete the system
eqs_sc = scale(me_comp) # Rescales the equations using symmetries 
@named sys = System(eqs_sc)

## Unseeded solution (from notebook 2)

In [ ]:
data_dir = joinpath("..", "data", "Nc_Ng") # Relative to this notebook

n_0 = npzread(joinpath(data_dir, "n.npy"))
it = npzread(joinpath(data_dir, "iterations.npy"))

## Steady state with seed

In [ ]:
coh = zeros(num_Nc,num_Nl) # |<a_s>|
sq = ones(num_Nc,num_Nl) # ΔX² (normalised to vacuum)

seeds = LinRange(0.1,3.0,40) # Seed amplitudes, in units of sqrt(n_0)(κ+κ_f)/2

it_seed = zeros(num_Nc,num_Nl)
dist_seed = 10*ones(num_Nc,num_Nl)
dist2_seed = zeros(num_Nc,num_Nl)

param_list = (Δ_cav_b, Δ_0_b, g_b, G, χ_b, κ, κ_b, R, Nc, Nl, pc, γ_ϕc, γc, pl, γ_ϕl, γl, Λ)

progress = Progress(num_Nc*num_Nl)
sim_lock = ReentrantLock()

Threads.@threads for c in 1:num_Nc

    for l in 1:num_Nl

        # Skip the non-stationary points of the unseeded run
        if it[c,l]>10
            coh[c,l] = NaN
            sq[c,l] = NaN
            next!(progress)
            continue
        end

        ind_seed = 1
        u0 = zeros(ComplexF64,length(eqs_sc))

        # Increase the seed until a coherent solution, |<a_s>| > 1.2 sqrt(n_0), is found
        while coh[c,l]/sqrt(n_0[c,l]) <= 1.2 && ind_seed<length(seeds)+1

            dist_seed[c,l] = 10
            dist2_seed[c,l] = 1
            it_seed[c,l] = 0
            sol = 0

            while (dist_seed[c,l]>1 || dist2_seed[c,l]>0.01) && it_seed[c,l]<10

                p0 = (w_c_b[c,l], w_0_b[c,l], g[c,l], sqrt(g_02[c,l]), chi_b[c,l], kappa, kappa_b, r, N_atom_I[c], N_atom_II[l], P_eff[c,l], dep_eff[c,l]+dep_eff_b[c,l], dec_eff[c,l]+dec_eff_b[c,l], P_T[c,l], dep_T[c,l], dec_eff[c,l], seeds[ind_seed]*sqrt(n_0[c,l])*(dec_f[c,l]+kappa)/2)
                dict = merge(Dict(unknowns(sys) .=> u0), Dict(param_list .=> p0))

                prob = ODEProblem(sys,dict,(0.0,30000.0))
                sol = solve(prob, Tsit5(); abstol=1e-8, reltol=1e-8, maxiters=1e6)

                lock(sim_lock) do
                    dist_new = maximum(abs.(sol[as'as])) - minimum(abs.(sol[as'as]))
                    dist2_seed[c,l] = maximum(abs.(sol[as*as])) - minimum(abs.(sol[as*as]))

                    if abs(dist_new - dist_seed[c,l])/dist_seed[c,l]<0.02 && dist_seed[c,l] > 10
                        dist_seed[c,l] = dist_new
                        it_seed[c,l] = 10
                    else
                        dist_seed[c,l] = dist_new
                    end
                end

                u0 = sol.u[end]

                it_seed[c,l] = it_seed[c,l]+1

                if sol.retcode != ReturnCode.Success
                    dist_seed[c,l] = 0.0
                    break
                end
            end

            if sol.retcode != ReturnCode.Success
                ind_seed += 1
                coh[c,l] = NaN
                sq[c,l] = NaN
                continue
            end

            n_t = length(sol.t)

            lock(sim_lock) do
                coh[c,l] = sum(abs.(sol[as]))/n_t
                sq[c,l] = sum((real.(sol[as*as]).-real.(sol[as]).^2 .+real.(sol[as'*as']).-real.(sol[as']).^2 .+2*real.(sol[as'*as]).+1 .-2*real.(sol[as]).*real.(sol[as'])).*exp.(-2*r))/n_t
            end

            ind_seed += 1
        end

        next!(progress)
    end
end

## Save

In [ ]:
npzwrite(joinpath(data_dir, "dX2_seed.npy"), sq)